# 1. 基于 BCEmbedding

## 1.1 调用EmbeddingModel计算句向量表示

In [1]:
from BCEmbedding import EmbeddingModel

/root/miniconda3/envs/bce/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sentences = ['今天天气不错哟', '明天一起去徒步']

In [3]:
model = EmbeddingModel(model_name_or_path="bce-embedding-base_v1/")

03/05/2024 06:34:44 - [INFO] -BCEmbedding.models.EmbeddingModel->>>    Loading from `bce-embedding-base_v1/`.
03/05/2024 06:34:45 - [INFO] -BCEmbedding.models.EmbeddingModel->>>    Execute device: cuda;	 gpu num: 1;	 use fp16: False;	 embedding pooling type: cls;	 trust remote code: False


In [4]:
embeddings = model.encode(sentences)

Extract embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.55it/s]


In [5]:
embeddings.shape

(2, 768)

## 1.2 调用RerankerModel计算句子对的语义相关分数

In [6]:
from BCEmbedding import RerankerModel

query = "一个女人站在高崖上单腿站立，俯瞰一条河流。"

passages = ["一个女人站在悬崖上。",
            "一个孩子在她的卧室里读书。"]

# 构造语句对
sentence_pairs = [[query, passage] for passage in passages]

# 初始化 reranker 模型
rerank_model = RerankerModel(model_name_or_path="./bce-reranker-base_v1/")

# （1）计算语句对的相似性得分
scores = rerank_model.compute_score(sentence_pairs)

# (2) 对passages排序
rerank_results = rerank_model.rerank(query, passages)

print(rerank_results)

03/05/2024 06:34:48 - [INFO] -BCEmbedding.models.RerankerModel->>>    Loading from `./bce-reranker-base_v1/`.
03/05/2024 06:34:48 - [INFO] -BCEmbedding.models.RerankerModel->>>    Execute device: cuda;	 gpu num: 1;	 use fp16: False
Calculate scores: 100%|██████████| 1/1 [00:00<00:00, 15.61it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


{'rerank_passages': ['一个女人站在悬崖上。', '一个孩子在她的卧室里读书。'], 'rerank_scores': [0.7433750033378601, 0.38531553745269775], 'rerank_ids': [0, 1]}


# 2. 基于 transformers

## 2.1 调用EmbeddingModel计算句向量表示

In [7]:
from transformers import AutoModel, AutoTokenizer
import torch

passages = ["一个女人站在悬崖上。",
            "一个孩子在她的卧室里读书。"]

# 初始化模型
tokenizer = AutoTokenizer.from_pretrained("./bce-embedding-base_v1/")
model = AutoModel.from_pretrained("./bce-reranker-base_v1/")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

# 基于tokenizer进行分词
encoded_inputs = tokenizer(passages, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)

# 获取embedding
outputs = model(**encoded_inputs, return_dict=True)

embeddings = outputs.last_hidden_state[:, 0] # cls

embeddings = embeddings / embeddings.norm(dim=1, keepdim=True) # 归一化

print(embeddings.shape)

Some weights of XLMRobertaModel were not initialized from the model checkpoint at ./bce-reranker-base_v1/ and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


torch.Size([2, 768])


## 2.2 调用RerankerModel计算句子对的语义相关分数

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 初始化模型
tokenizer = AutoTokenizer.from_pretrained("./bce-reranker-base_v1/")
model = AutoModelForSequenceClassification.from_pretrained("./bce-reranker-base_v1/")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

query = "一个女人站在高崖上单腿站立，俯瞰一条河流。"
passages = ["一个女人站在悬崖上。",
            "一个孩子在她的卧室里读书。"]
# 构造语句对
sentence_pairs = [[query, passage] for passage in passages]

# 获取分词后的输入
encoded_inputs = tokenizer(sentence_pairs, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)

# ① 通过模型计算每个语句对的分类得分（logits）,return_dict=True指示模型返回一个包含各种输出的字典，
# ② .logits提取出了分类得分，logits是模型的原始输出，对于序列分类任务，这是一个形状为(batch_size, num_labels)的张量，其中每个条目表示对应类别的得分。
# ③ .view(-1,): 这个操作改变logits张量的形状。-1意味着该维度的大小会自动计算，使得结果是一个一维张量。
scores = model(**encoded_inputs, return_dict=True).logits.view(-1,).float()

# ④ 使用sigmoid函数将每个得分转换为一个介于0和1之间的值，可以解释为概率。
#    对于二分类任务，sigmoid函数非常适合，因为它能够将任何实数映射到0和1之间，从而表示模型对每个类别的置信度。
scores = torch.sigmoid(scores)

print(scores)

tensor([0.7631, 0.3893], device='cuda:0', grad_fn=<SigmoidBackward0>)


# 3. 基于 sentence_transformers

## 3.1 调用EmbeddingModel计算句向量表示¶

In [9]:
from sentence_transformers import SentenceTransformer

passages = ["一个女人站在悬崖上。",
            "一个孩子在她的卧室里读书。"]

model = SentenceTransformer("./bce-embedding-base_v1/")

embeddings = model.encode(passages, normalize_embeddings=True)

print(embeddings.shape)

03/05/2024 06:34:56 - [INFO] -sentence_transformers.SentenceTransformer->>>    Load pretrained SentenceTransformer: ./bce-embedding-base_v1/
03/05/2024 06:34:59 - [INFO] -sentence_transformers.SentenceTransformer->>>    Use pytorch device_name: cuda
Batches: 100%|██████████| 1/1 [00:00<00:00, 34.84it/s]

(2, 768)


## 3.2 调用RerankerModel计算句子对的语义相关分数

In [10]:
from sentence_transformers import CrossEncoder

model = CrossEncoder("./bce-reranker-base_v1/", max_length=512)


query = "一个女人站在高崖上单腿站立，俯瞰一条河流。"
passages = ["一个女人站在悬崖上。",
            "一个孩子在她的卧室里读书。"]
# 构造语句对
sentence_pairs = [[query, passage] for passage in passages]

scores = model.predict(sentence_pairs)

print(scores)

03/05/2024 06:35:02 - [INFO] -sentence_transformers.cross_encoder.CrossEncoder->>>    Use pytorch device: cuda
Batches: 100%|██████████| 1/1 [00:00<00:00,  2.54it/s]

[0.76313466 0.3892789 ]
